#### Corpus construction + weak labeling
Download all abstracts, dictionary-match adjuvants, then later drop PMIDs that don’t match.
1. Build the master PMID list (NO filtering yet)
2. Download ALL abstracts from PubMed
3. Dictionary match (EXACT + EXPANDED FORMS)
4. Coverage & drop analysis
    (If there are PMIDs that don’t match adjuvant and come from pathogen table, drop them later.)

In [1]:
import pandas as pd
import ast

from dotenv import load_dotenv
load_dotenv()  # loads .env from the notebook’s directory

# -------------------------
# Load KB
# -------------------------
kb_df = pd.read_csv(
    "Dataset/VIOLIN_12-10-2025/interim/adjuvant_kb.csv", encoding="utf-8-sig"
)

# -------------------------
# Helper: parse list columns safely
# -------------------------
def parse_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.startswith("["):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

kb_df["pmids"] = kb_df["pmids"].apply(parse_list)

# -------------------------
# Build master PMID list
# -------------------------
pmids = set()

for lst in kb_df["pmids"]:
    pmids.update(lst)

pmids = sorted(pmids)

# -------------------------
# Save for PubMed download
# -------------------------
out_file = "Dataset/VIOLIN_12-10-2025/interim/pmids_all.txt"
with open(out_file, "w", encoding="utf-8") as f:
    for p in pmids:
        f.write(f"{p}\n")

print(f"✅ Total PMIDs to download: {len(pmids)}")
print(f"✅ Saved to: {out_file}")


✅ Total PMIDs to download: 1341
✅ Saved to: Dataset/VIOLIN_12-10-2025/interim/pmids_all.txt


In [2]:
#%pip install biopython tqdm


#### point of view: We processed 1,341 PubMed abstracts in JSONL format and applied a dictionary-based NER pipeline using a curated adjuvant lexicon.

In [3]:
from Bio import Entrez
from tqdm import tqdm
import time
import json
import os

# ============================
# CONFIG — UPDATE THESE
# ============================
Entrez.email = os.getenv("ENTREZ_EMAIL")
Entrez.api_key = os.getenv("ENTREZ_API_KEY")
if not Entrez.email or not Entrez.api_key:
    raise ValueError("Set ENTREZ_EMAIL and ENTREZ_API_KEY env vars before running.")

PMID_FILE = "Dataset/VIOLIN_12-10-2025/interim/pmids_all.txt"
OUT_JSONL = "Dataset/VIOLIN_12-10-2025/interim/pubmed_abstracts.jsonl"

BATCH_SIZE = 200        # Safe with API key
SLEEP_SECONDS = 0.2     # Extra safety

# ============================
# LOAD PMIDs
# ============================
with open(PMID_FILE, encoding="utf-8") as f:
    pmids = [line.strip() for line in f if line.strip()]

print(f"✅ Loaded {len(pmids)} PMIDs")

# ============================
# HELPERS
# ============================
def fetch_batch(pmid_batch):
    handle = Entrez.efetch(
        db="pubmed",
        id=",".join(pmid_batch),
        rettype="abstract",
        retmode="xml",
    )
    return Entrez.read(handle)

def extract_text(article):
    pmid = article["MedlineCitation"]["PMID"]

    article_data = article["MedlineCitation"]["Article"]
    title = article_data.get("ArticleTitle", "")

    abstract = ""
    if "Abstract" in article_data:
        parts = article_data["Abstract"].get("AbstractText", [])
        abstract = " ".join(str(p) for p in parts)

    text = f"{title} {abstract}".strip()

    return {
        "pmid": str(pmid),
        "title": title,
        "abstract": abstract,
        "text": text
    }

# ============================
# DOWNLOAD LOOP
# ============================
os.makedirs(os.path.dirname(OUT_JSONL), exist_ok=True)

written = 0

with open(OUT_JSONL, "w", encoding="utf-8") as out:
    for i in tqdm(range(0, len(pmids), BATCH_SIZE)):
        batch = pmids[i:i+BATCH_SIZE]

        try:
            records = fetch_batch(batch)
        except Exception as e:
            print(f"⚠️ Error fetching batch {i}-{i+BATCH_SIZE}: {e}")
            time.sleep(5)
            continue

        articles = records.get("PubmedArticle", [])
        for art in articles:
            obj = extract_text(art)
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
            written += 1

        time.sleep(SLEEP_SECONDS)

print(f"✅ Saved {written} PubMed records to JSONL")
print(f"📄 Output: {OUT_JSONL}")


✅ Loaded 1341 PMIDs


100%|██████████| 7/7 [00:14<00:00,  2.06s/it]

✅ Saved 1341 PubMed records to JSONL
📄 Output: Dataset/VIOLIN_12-10-2025/interim/pubmed_abstracts.jsonl


In [4]:
import json

with open("Dataset/VIOLIN_12-10-2025/interim/pubmed_abstracts.jsonl", encoding="utf-8") as f:
    for _ in range(3):
        print(json.loads(next(f)))


{'pmid': '10024591', 'title': 'Effect of transforming growth factor beta on experimental Salmonella typhimurium infection in mice.', 'abstract': 'We have investigated the effect of the in vivo administration of recombinant transforming growth factor beta (rTGF-beta) on the pathogenic mechanisms involved in Salmonella typhimurium experimental infection in mice. The protective response elicited by macrophages was induced by rTGF-beta1 by 2 days after experimental infection, as demonstrated by an increased NO production, while the humoral protective effect began with cytokine mRNA expression 2 days after the challenge and continued after 5 days with cytokine release and lymphocyte activation. We demonstrated that all mice who received rTGF-beta1 survived 7 days after infection. The number of bacteria recovered in the spleens and in the livers of rTGF-beta1-treated mice 2 and 5 days after infection was significantly smaller than that found in the same organs after phosphate-buffered saline